[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_10/MFM_ch10_seminar/MFM_ch10_seminar.ipynb)

# MFM_ch10_seminar

Seminar 10 computations: moments, bias and standard error of the Roll estimator, Glosten-Milgrom for any prior and learning speed, the Kyle equilibrium and lambda as a regression slope, Almgren-Chriss by Euler-Lagrange, the PIN likelihood in stable form; intraday U-shape with day-block bootstrap, spread estimators at two frequencies, spread and Amihud measures across 25 assets, illiquidity and the VIX with HAC errors, the volume-price relation with an instrumental variable, the Bitcoin clock, price discovery between the BET ETF and BET-TR, and the time-series illiquidity premium on the BVB, US and crypto markets.

*Modelling Financial Markets (MFM), Chapter 10: Market Microstructure. Author: Daniel Traian Pele.*

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Gray     = '#7F7F7F'   # doar linii de referinta
GROUP_COL = {'US': MainBlue, 'BVB': IDAred, 'Crypto': Amber}

HERE = '.'
SEED = 42
START2 = '2024-09-19'
START10 = '2016-09-19'
B_BOOT = 1000


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, handles, labels, ncol=3, y=0.0):
    """O singura legenda pentru o figura cu mai multe panouri, sub figura."""
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# cheie -> (simbol, eticheta, grup)
ASSETS = {
    'SPY': ('SPY.US', 'SPY', 'US'), 'AAPL': ('AAPL.US', 'Apple', 'US'), 'MSFT': ('MSFT.US', 'Microsoft', 'US'),
    'NVDA': ('NVDA.US', 'NVIDIA', 'US'), 'JPM': ('JPM.US', 'JPMorgan', 'US'), 'GME': ('GME.US', 'GameStop', 'US'),
    'MSTR': ('MSTR.US', 'Strategy', 'US'),
    'TLV': ('TLV.RO', 'Banca Transilvania', 'BVB'), 'SNP': ('SNP.RO', 'OMV Petrom', 'BVB'),
    'H2O': ('H2O.RO', 'Hidroelectrica', 'BVB'), 'BRD': ('BRD.RO', 'BRD', 'BVB'), 'SNG': ('SNG.RO', 'Romgaz', 'BVB'),
    'SNN': ('SNN.RO', 'Nuclearelectrica', 'BVB'), 'TGN': ('TGN.RO', 'Transgaz', 'BVB'),
    'EL': ('EL.RO', 'Electrica', 'BVB'), 'DIGI': ('DIGI.RO', 'Digi', 'BVB'), 'TEL': ('TEL.RO', 'Transelectrica', 'BVB'),
    'TVBETETF': ('TVBETETF.RO', 'BET ETF', 'BVB'),
    'BTC': ('BTC-USD.CC', 'Bitcoin', 'Crypto'), 'ETH': ('ETH-USD.CC', 'Ethereum', 'Crypto'),
    'SOL': ('SOL-USD.CC', 'Solana', 'Crypto'), 'XRP': ('XRP-USD.CC', 'XRP', 'Crypto'),
    'DOGE': ('DOGE-USD.CC', 'Dogecoin', 'Crypto'), 'ADA': ('ADA-USD.CC', 'Cardano', 'Crypto'),
    'LTC': ('LTC-USD.CC', 'Litecoin', 'Crypto'),
}
GROUPS = {g: [k for k, v in ASSETS.items() if v[2] == g] for g in ('US', 'BVB', 'Crypto')}
LABELS = {k: v[1] for k, v in ASSETS.items()}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    if symbol in _CACHE:
        return _CACHE[symbol]
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    _CACHE[symbol] = pd.read_csv(src, index_col='date', parse_dates=True).sort_index()
    return _CACHE[symbol]


def read_reference_rate(currency='USD', start='2014-01-01', end=END):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = ('REF', currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'rate']).drop_duplicates('date').set_index('date')['rate']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def _clean(key, d):
    d = d[(d['high'] > d['low']) & (d['volume'].fillna(0) > 0)]
    if ASSETS[key][2] != 'Crypto':
        d = d[d.index.dayofweek < 5]
        # cotatii eronate izolate: pretul ajustat de peste doua ori mai mare / mai mic decat mediana locala (21 de zile)
        med = d['adjusted_close'].rolling(21, center=True, min_periods=5).median()
        d = d[np.abs(np.log(d['adjusted_close'] / med)) < np.log(2)]
    return d


def ohlc(key, start='2016-09-19', end=END):
    """Deschidere, maxim, minim, inchidere ajustate cu factorul pretului ajustat (dividende si split-uri)."""
    d = read_market(ASSETS[key][0]).loc[start:end].copy()
    f = d['adjusted_close'] / d['close']
    for c in ['open', 'high', 'low', 'close']:
        d[c] = d[c] * f
    return _clean(key, d)[['open', 'high', 'low', 'close', 'volume']]


def split_adjusted_close(d):
    """Pretul de inchidere ajustat DOAR pentru split-uri (aceeasi baza ca volumul in numar de actiuni)."""
    k = (d['close'].shift(1) / d['close']) / (d['adjusted_close'].shift(1) / d['adjusted_close'])
    k = k.where(np.abs(np.log(k)) > 0.15, 1.0).fillna(1.0)      # zilele de split: raportul de divizare
    back = k[::-1].cumprod()[::-1].shift(-1).fillna(1.0)          # produsul split-urilor ulterioare fiecarei zile
    return d['close'] / back


def returns(key, start='2016-09-19', end=END):
    """Randamente log zilnice din pretul ajustat (zilele fara tranzactii eliminate)."""
    d = read_market(ASSETS[key][0]).loc[start:end]
    d = _clean(key, d)
    return np.log(d['adjusted_close']).diff().dropna().rename(key)


def dollar_volume(key, start='2016-09-19', end=END):
    """Valoarea tranzactionata zilnic, in USD."""
    d = read_market(ASSETS[key][0]).loc[:end]
    grp = ASSETS[key][2]
    if grp == 'Crypto':
        dv = d['volume']                                          # deja in USD
    else:
        dv = split_adjusted_close(d) * d['volume']
        if grp == 'BVB':                                          # RON -> USD la cursul de referinta BNR
            fx = read_reference_rate('USD', start='2014-01-01', end=end)
            dv = dv / fx.reindex(dv.index).ffill()
    d = _clean(key, d.assign(dv=dv))
    return d['dv'].loc[start:end].dropna().rename(key)


def intraday_spy():
    """Bare de 5 minute SPY, sesiunea regulata (09:30-15:55 = inceputul barei), doar zilele complete (78 de bare)."""
    fname = 'intraday/SPY.US_5m.csv'
    path = os.path.join(MARKET_DIR, fname)
    d = pd.read_csv(path if os.path.exists(path) else REPO_RAW + fname, parse_dates=['datetime_utc'])
    t = d['datetime_utc'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None)
    d = d.assign(t=t, date=t.dt.normalize(), tod=t.dt.strftime('%H:%M')).drop(columns='datetime_utc')
    d = d[(d['tod'] >= '09:30') & (d['tod'] <= '15:55')]
    n = d.groupby('date').size()
    d = d[d['date'].isin(n[n == 78].index)].set_index('t').sort_index()
    d['r'] = np.log(d['close']).groupby(d['date']).diff()
    first = d['tod'] == '09:30'
    d.loc[first, 'r'] = np.log(d.loc[first, 'close'] / d.loc[first, 'open'])   # prima bara: de la deschidere
    d['dv'] = d['close'] * d['volume']
    return d


def intraday_btc():
    """Bare de 5 minute Bitcoin (UTC), 24 de ore din 24; volumul nu este folosit (lipseste pe multe bare)."""
    fname = 'intraday/BTC-USD.CC_5m.csv'
    path = os.path.join(MARKET_DIR, fname)
    d = pd.read_csv(path if os.path.exists(path) else REPO_RAW + fname, parse_dates=['datetime_utc'])
    d = d.set_index('datetime_utc').sort_index()
    d = d[~d.index.duplicated()]
    d['date'] = d.index.normalize()
    d['r'] = np.log(d['close']).diff()
    gap = d.index.to_series().diff() != pd.Timedelta('5min')
    d.loc[gap, 'r'] = np.nan                                       # fara randamente peste goluri
    return d

## Estimators and models

In [ ]:
def roll_spread(close):
    """Spread-ul relativ Roll din covarianta de ordinul 1 a randamentelor log; NaN daca covarianta este pozitiva."""
    r = np.log(np.asarray(close, float))
    dp = np.diff(r)
    c = np.mean((dp[1:] - dp.mean()) * (dp[:-1] - dp.mean()))
    return (2 * np.sqrt(-c) if c < 0 else np.nan), c


def cs_spread(high, low):
    """Estimatorul Corwin-Schultz pentru fiecare pereche de perioade consecutive (valorile negative devin 0)."""
    h, l = np.log(np.asarray(high, float)), np.log(np.asarray(low, float))
    beta = (h[:-1] - l[:-1]) ** 2 + (h[1:] - l[1:]) ** 2
    gamma = (np.maximum(h[:-1], h[1:]) - np.minimum(l[:-1], l[1:])) ** 2
    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    return np.clip(2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha)), 0, None)


def ar_terms(close, high, low):
    """Termenii Abdi-Ranaldo: 4 (c_t - eta_t)(c_t - eta_{t+1}); media lor estimeaza s^2."""
    c = np.log(np.asarray(close, float))
    eta = (np.log(np.asarray(high, float)) + np.log(np.asarray(low, float))) / 2
    return 4 * (c[:-1] - eta[:-1]) * (c[:-1] - eta[1:])


def ar_spread(close, high, low):
    """Spread-ul relativ Abdi-Ranaldo: sqrt(max(media termenilor, 0))."""
    m = np.mean(ar_terms(close, high, low))
    return np.sqrt(max(m, 0.0)), m


def edge_spread(open_, high, low, close, sign=True):
    """EDGE (Ardia, Guidotti si Kroencke, 2024, JFE 161, 103916): spread-ul relativ din preturile OHLC.
    Transcrierea implementarii de referinta a autorilor (pachetul bidask, licenta MIT). sign=True pastreaza semnul
    estimarii (recomandat cand se face media estimarilor pe mai multe ferestre: media ramane nedeplasata)."""
    o, h, l, c = (np.log(np.asarray(x, float)) for x in (open_, high, low, close))
    if len(o) < 3:
        return np.nan
    m = (h + l) / 2.
    h1, l1, c1, m1 = h[:-1], l[:-1], c[:-1], m[:-1]
    o, h, l, c, m = o[1:], h[1:], l[1:], c[1:], m[1:]
    r1, r2, r3, r4, r5 = m - o, o - m1, m - c1, c1 - m1, o - c1
    tau = np.where(np.isnan(h) | np.isnan(l) | np.isnan(c1), np.nan, (h != l) | (l != c1))
    po1 = tau * np.where(np.isnan(o) | np.isnan(h), np.nan, o != h)
    po2 = tau * np.where(np.isnan(o) | np.isnan(l), np.nan, o != l)
    pc1 = tau * np.where(np.isnan(c1) | np.isnan(h1), np.nan, c1 != h1)
    pc2 = tau * np.where(np.isnan(c1) | np.isnan(l1), np.nan, c1 != l1)
    pt = np.nanmean(tau)
    po = np.nanmean(po1) + np.nanmean(po2)
    pc = np.nanmean(pc1) + np.nanmean(pc2)
    if np.nansum(tau) < 2 or po == 0 or pc == 0:
        return np.nan
    d1 = r1 - np.nanmean(r1) / pt * tau
    d3 = r3 - np.nanmean(r3) / pt * tau
    d5 = r5 - np.nanmean(r5) / pt * tau
    x1 = -4. / po * d1 * r2 - 4. / pc * d3 * r4
    x2 = -4. / po * d1 * r5 - 4. / pc * d5 * r4
    e1, e2 = np.nanmean(x1), np.nanmean(x2)
    v1, v2 = np.nanmean(x1 ** 2) - e1 ** 2, np.nanmean(x2 ** 2) - e2 ** 2
    vt = v1 + v2
    s2 = (v2 * e1 + v1 * e2) / vt if vt > 0 else (e1 + e2) / 2.
    s = np.sqrt(np.abs(s2))
    return float(s * np.sign(s2)) if sign else float(s)


def amihud(r, dv, scale=1e6):
    """Iliciditatea Amihud: media |r| (puncte de baza) la 1 milion USD tranzactionat."""
    x = pd.concat([r.rename('r'), dv.rename('dv')], axis=1, join='inner').dropna()
    x = x[x['dv'] > 0]
    return float((1e4 * x['r'].abs() / (x['dv'] / scale)).mean())


def daily_estimates(d):
    """Estimatorii de spread pe o fereastra de bare (o zi de bare intraday sau o perioada de date zilnice)."""
    rs, cov = roll_spread(d['close'])
    cs = cs_spread(d['high'], d['low'])
    ar2 = ar_terms(d['close'], d['high'], d['low'])
    return pd.Series(dict(cov=cov, roll=rs, cs=cs.mean(), ar2=ar2.mean()))


def roll_mc(c, sigma, T=78, R=20000, seed=42):
    """Harris (1990): simulam modelul Roll (m_t mers aleator cu pasi N(0, sigma^2), q_t = +-1 cu prob. 1/2)
    pe T preturi si calculam covarianta de ordinul 1 a variatiilor (demediate), ca pe date reale."""
    rng = np.random.default_rng(seed)
    p = np.cumsum(rng.normal(0, sigma, (R, T)), axis=1) + c * np.where(rng.random((R, T)) < 0.5, 1, -1)
    dp = np.diff(p, axis=1)
    dm = dp - dp.mean(axis=1, keepdims=True)
    return (dm[:, 1:] * dm[:, :-1]).mean(axis=1)


def price_discovery(y, p=1):
    """VECM bivariat cu vectorul de cointegrare (1, -1) impus: dy_t = a0 + alpha (y1 - y2)_{t-1} + sum Gamma_i dy_{t-i} + e_t.
    Intoarce alpha, statisticile t, ponderea componentei Gonzalo-Granger (alpha_perp normalizat) si limitele
    ponderii informationale Hasbrouck (cele doua ordonari Cholesky)."""
    y = np.asarray(y, float)
    dy = np.diff(y, axis=0)
    z = (y[:, 0] - y[:, 1])[:-1]
    X = np.array([[1.0, z[t]] + [v for i in range(1, p + 1) for v in dy[t - i]] for t in range(p, len(dy))])
    Y = dy[p:]
    B = np.linalg.lstsq(X, Y, rcond=None)[0]
    U = Y - X @ B
    Om = U.T @ U / (len(U) - X.shape[1])
    alpha = B[1]
    se = np.sqrt(np.linalg.inv(X.T @ X)[1, 1] * np.diag(Om))
    ap = np.array([-alpha[1], alpha[0]])                       # alpha_perp: alpha' alpha_perp = 0
    cs = ap / ap.sum()
    psi = ap                                                    # randul comun al impactului pe termen lung (pana la o scala)
    IS = []
    for order in ([0, 1], [1, 0]):
        F = np.linalg.cholesky(Om[np.ix_(order, order)])
        v = (psi[order] @ F) ** 2 / (psi @ Om @ psi)
        IS.append(v[np.argsort(order)])
    IS = np.array(IS)
    return dict(alpha=alpha, t=alpha / se, cs=cs, is_lo=IS.min(axis=0), is_hi=IS.max(axis=0),
                corr=float(Om[0, 1] / np.sqrt(Om[0, 0] * Om[1, 1])), resid=U, n=len(U))


def pin_loglik(B, S, alpha, delta, mu, eb, es):
    """Log-verosimilitatea EKOP pentru o zi cu B cumparari si S vanzari, in forma factorizata care evita
    depasirea numerica (Lin si Ke, 2011): ln L = -eb - es + B ln(mu + eb) + S ln(mu + es) - ln B! - ln S!
    + ln[(1 - alpha) xb^B xs^S + alpha delta e^-mu xb^B + alpha (1 - delta) e^-mu xs^S], xb = eb / (mu + eb)."""
    from scipy.special import gammaln, logsumexp
    lxb, lxs = np.log(eb / (mu + eb)), np.log(es / (mu + es))
    terms = [np.log(1 - alpha) + B * lxb + S * lxs, np.log(alpha * delta) - mu + B * lxb,
             np.log(alpha * (1 - delta)) - mu + S * lxs]
    base = -eb - es + B * np.log(mu + eb) + S * np.log(mu + es) - gammaln(B + 1) - gammaln(S + 1)
    return float(base + logsumexp(terms)), terms


def walk_book(asks, qty):
    """Ordin de cumparare la piata de marime qty pe ofertele de vanzare asks = [(pret, cantitate), ...] crescator.
    Intoarce executiile, pretul mediu ponderat si ofertele ramase."""
    fills, left, rest = [], qty, []
    for p, q in asks:
        if left <= 0:
            rest.append((p, q))
            continue
        take = min(q, left)
        fills.append((p, take))
        left -= take
        if q > take:
            rest.append((p, q - take))
    vwap = sum(p * q for p, q in fills) / sum(q for _, q in fills)
    return fills, vwap, rest, left


def gm_quotes(theta, mu, vl, vh):
    """Cotatiile ask si bid: asteptarea valorii conditionat de o cumparare, respectiv de o vanzare.
    theta = P(V = vh), mu = ponderea investitorilor informati; cei neinformati cumpara sau vand cu prob. 1/2."""
    pb_h, pb_l = mu + (1 - mu) / 2, (1 - mu) / 2               # P(cumparare | V = vh), P(cumparare | V = vl)
    th_b = pb_h * theta / (pb_h * theta + pb_l * (1 - theta))   # P(V = vh | cumparare)
    th_s = pb_l * theta / (pb_l * theta + pb_h * (1 - theta))   # P(V = vh | vanzare)
    return vl + th_b * (vh - vl), vl + th_s * (vh - vl), th_b, th_s


def gm_simulate(mu, vl=90.0, vh=110.0, v=110.0, n=60, seed=1):
    """Un sir de tranzactii cand valoarea adevarata este v; formatorul de piata invata din fluxul de ordine."""
    rng = np.random.default_rng(seed)
    theta, rows = 0.5, []
    for t in range(n):
        ask, bid, th_b, th_s = gm_quotes(theta, mu, vl, vh)
        informed = rng.random() < mu
        buy = (v == vh) if informed else (rng.random() < 0.5)
        rows.append(dict(t=t, ask=ask, bid=bid, theta=theta, buy=buy))
        theta = th_b if buy else th_s
    return pd.DataFrame(rows)


def kyle(sigma_v, sigma_u):
    """Echilibrul Kyle: v ~ N(p0, sigma_v^2), cererea zgomotoasa u ~ N(0, sigma_u^2).
    Strategia informatului x = beta (v - p0), pretul p = p0 + lambda (x + u)."""
    beta = sigma_u / sigma_v
    lam = sigma_v / (2 * sigma_u)
    profit = sigma_v * sigma_u / 2
    return dict(beta=beta, lam=lam, profit=profit, post_var=sigma_v ** 2 / 2)


def ac_trajectory(X, T, N, sigma, eta, gamma, lam, eps=0.0):
    """Traiectoria optima de lichidare (formulele exacte in timp discret).
    X actiuni de vandut in T zile, N intervale; sigma = volatilitatea pretului (USD / zi^0.5);
    impact temporar h(v) = eps sgn + eta v; impact permanent g(v) = gamma v; lam = aversiunea la risc."""
    tau = T / N
    eta_t = eta - gamma * tau / 2
    t = np.arange(N + 1) * tau
    if lam == 0:
        x = X * (1 - t / T)
    else:
        kt2 = lam * sigma ** 2 / eta_t                       # kappa_tilde^2
        kappa = np.arccosh(kt2 * tau ** 2 / 2 + 1) / tau
        x = X * np.sinh(kappa * (T - t)) / np.sinh(kappa * T)
    n = -np.diff(x)
    E = 0.5 * gamma * X ** 2 + eps * X + eta_t / tau * np.sum(n ** 2)
    V = sigma ** 2 * tau * np.sum(x[1:] ** 2)
    return t, x, E, V


def ac_frontier(X, T, N, sigma, eta, gamma, lams):
    """Frontiera eficienta (abaterea standard, costul asteptat) pentru mai multe aversiuni la risc."""
    out = []
    for lam in lams:
        _, _, E, V = ac_trajectory(X, T, N, sigma, eta, gamma, lam)
        out.append((lam, E, np.sqrt(V)))
    return pd.DataFrame(out, columns=['lam', 'E', 'sd'])


def zi_simulate(n_events=300_000, L=40, rate_limit=1.0, rate_market=0.25, rate_cancel=0.025, n_ticks=4000, seed=7):
    """Ordine limita (uniform pe L tick-uri de la cotatia opusa), ordine la piata si anulari, toate de o unitate.
    Intoarce seria (mijloc, spread) si instantanee ale registrului."""
    rng = np.random.default_rng(seed)
    bid_q = np.zeros(n_ticks, int)
    ask_q = np.zeros(n_ticks, int)
    mid0 = n_ticks // 2
    bid_q[mid0 - 20:mid0] = 3
    ask_q[mid0 + 1:mid0 + 21] = 3
    rows, snaps, offset = [], [], 0
    for e in range(n_events):
        bb = np.flatnonzero(bid_q)[-1]
        ba = np.flatnonzero(ask_q)[0]
        n_orders = bid_q.sum() + ask_q.sum()
        w = np.array([2 * L * rate_limit, 2 * rate_market, rate_cancel * n_orders])
        u = rng.random() * w.sum()
        side = rng.random() < 0.5
        if u < w[0]:                                   # ordin limita
            k = rng.integers(1, L + 1)
            if side:
                p = ba - k
                bid_q[p] += 1
            else:
                p = bb + k
                ask_q[p] += 1
        elif u < w[0] + w[1]:                          # ordin la piata
            if side:
                ask_q[ba] -= 1 if ask_q[ba] > 0 else 0
            else:
                bid_q[bb] -= 1 if bid_q[bb] > 0 else 0
        else:                                          # anulare a unui ordin existent, ales uniform
            if side and bid_q.sum() > 1:
                idx = rng.choice(np.flatnonzero(bid_q), p=bid_q[bid_q > 0] / bid_q.sum())
                bid_q[idx] -= 1
            elif (not side) and ask_q.sum() > 1:
                idx = rng.choice(np.flatnonzero(ask_q), p=ask_q[ask_q > 0] / ask_q.sum())
                ask_q[idx] -= 1
        if not bid_q.any():
            bid_q[bb] = 1
        if not ask_q.any():
            ask_q[ba] = 1
        bb = np.flatnonzero(bid_q)[-1]
        ba = np.flatnonzero(ask_q)[0]
        if bb < 5 * L or ba > n_ticks - 5 * L:         # recentrare: registrul ramane in mijlocul grilei
            sh = int((bb + ba) / 2 - n_ticks // 2)
            bid_q, ask_q = np.roll(bid_q, -sh), np.roll(ask_q, -sh)
            if sh > 0:
                bid_q[-sh:] = 0
                ask_q[-sh:] = 0
            else:
                bid_q[:-sh] = 0
                ask_q[:-sh] = 0
            offset += sh
        if e % 50 == 0:
            bb = np.flatnonzero(bid_q)[-1]
            ba = np.flatnonzero(ask_q)[0]
            rows.append((e, (bb + ba) / 2 - mid0 + offset, ba - bb))
        if e > n_events // 3 and e % 1000 == 0:
            bb = np.flatnonzero(bid_q)[-1]
            ba = np.flatnonzero(ask_q)[0]
            m = (bb + ba) / 2
            rel = np.arange(n_ticks) - m
            sel = np.abs(rel) <= 30
            snaps.append(pd.DataFrame({'rel': rel[sel], 'bid': bid_q[sel], 'ask': ask_q[sel]}))
    path = pd.DataFrame(rows, columns=['event', 'mid', 'spread'])
    return path, snaps, (bid_q, ask_q)

## Helper functions

In [ ]:
B = 1000

AC = {'X': 1000000.0, 'T': 5, 'N': 5, 'sigma': 0.95, 'eta': 2.5e-06, 'gamma': 2.5e-07}

C_MARKETS = {'BVB': {'ret': ('BETTR.INDX', 'close'), 'keys': ['TLV', 'SNP', 'H2O', 'BRD', 'SNG', 'SNN', 'TGN', 'EL', 'DIGI', 'TEL']}, 'US': {'ret': ('SPY.US', 'adjusted_close'), 'keys': ['AAPL', 'MSFT', 'NVDA', 'JPM', 'GME', 'MSTR']}, 'Crypto': {'ret': ('BTC-USD.CC', 'close'), 'keys': ['ETH', 'SOL', 'XRP', 'DOGE', 'ADA', 'LTC']}}

PIN_PAR = {'alpha': 0.3, 'delta': 0.5, 'mu': 400.0, 'eb': 1000.0, 'es': 1000.0}

END_PD = '2026-09-18'

PD_START = '2024-09-19'

def spy_tod(s):
    """Profilul pe intervale de 5 minute: media |r| (pb), volumul median (milioane de actiuni), ponderea in volumul zilei."""
    x = s.dropna(subset=['r'])
    tod = x.groupby('tod').agg(absr=('r', lambda v: 1e4 * v.abs().mean()), vol=('volume', 'median'))
    share = (s['volume'] / s.groupby('date')['volume'].transform('sum')).groupby(s['tod']).mean()
    tod['share'] = 100 * share
    return tod


def intraday_spreads(s):
    """Estimatorii Roll, Corwin-Schultz si Abdi-Ranaldo calculati pe barele de 5 minute ale fiecarei zile."""
    rows = {}
    for d, g in s.dropna(subset=['close', 'high', 'low']).groupby('date'):
        rs, cov = roll_spread(g['close'])
        rows[d] = dict(cov=cov, cs=cs_spread(g['high'], g['low']).mean(),
                       ar2=ar_terms(g['close'], g['high'], g['low']).mean(), price=g['close'].mean(),
                       edge=edge_spread(g['open'], g['high'], g['low'], g['close']),
                       var=np.var(np.diff(np.log(g['close'].values)), ddof=1))
    return pd.DataFrame(rows).T


def daily_spreads(key, start=START2):
    """Estimatorii din date zilnice pe ultimii doi ani (valori in puncte de baza)."""
    d = ohlc(key, start)
    rs, cov = roll_spread(d['close'])
    cs = cs_spread(d['high'], d['low'])
    ar2 = ar_terms(d['close'], d['high'], d['low'])
    return dict(roll=1e4 * rs if rs == rs else np.nan, roll_cov=float(cov), cs=1e4 * float(np.mean(cs)),
                edge=1e4 * edge_spread(d['open'], d['high'], d['low'], d['close']),
                ar=1e4 * float(np.sqrt(max(np.mean(ar2), 0))), ar_neg=bool(np.mean(ar2) <= 0), n=int(len(d)))


def spread_table():
    return {k: daily_spreads(k) for k in ASSETS}


def amihud_table(start=START2):
    out = {}
    for k in ASSETS:
        r = returns(k, start)
        dv = dollar_volume(k, start)
        out[k] = dict(illiq=amihud(r, dv), dv_med=float(dv.median() / 1e6), n=int(len(r)))
    return out


def spy_illiq_daily(s):
    """Iliciditatea intrazilnica: media |r_5min| (pb) la 100 milioane USD tranzactionati, pe zi."""
    x = s.dropna(subset=['r', 'dv'])
    x = x[x['dv'] > 0]
    return (1e4 * x['r'].abs() / (x['dv'] / 1e8)).groupby(x['date']).mean().rename('illiq')


def sqrt_relation(s, nb=20):
    x = s.dropna(subset=['r', 'volume']).copy()
    x = x[x['volume'] > 0]
    x['part'] = x['volume'] / x.groupby('tod')['volume'].transform('median')
    x['z'] = x['r'].abs() / x.groupby('date')['r'].transform('std')
    bins = np.quantile(x['part'], np.linspace(0, 1, nb + 1))
    g = x.groupby(pd.cut(x['part'], bins, include_lowest=True)).agg(v=('part', 'mean'), z=('z', 'mean'))
    slope, icpt = np.polyfit(np.log(g['v']), np.log(g['z']), 1)
    return x, g, slope, icpt


def roll_small_sample(E):
    """Covarianta Roll pe zi (77 de randamente de 5 minute): ponderea zilelor cu covarianta pozitiva, observata si
    simulata sub modelul Roll adevarat (spread de un pas de cotare; fara spread), si deplasarea din demediere."""
    n = 77
    v = E['var'].mean()
    c_tick = (0.01 / E['price']).mean() / 2
    out = dict(pos_obs=float((E['cov'] > 0).mean()), n_days=int(len(E)), sd_bar=float(1e4 * np.sqrt(v)),
               c_tick=float(1e4 * c_tick))
    for tag, c in [('tick', c_tick), ('zero', 0.0)]:
        sig = np.sqrt(v - 2 * c ** 2)
        cov = np.array([roll_mc(c, sig, T=n + 1, R=len(E), seed=SEED + b) for b in range(200)])
        pos = (cov > 0).mean(axis=1)                        # 200 de "esantioane" de lungimea selectiei reale
        pooled = 1e4 * 2 * np.sqrt(np.clip(-cov.mean(axis=1), 0, None))
        out[f'pos_{tag}'] = float(pos.mean())
        out[f'pos_{tag}_lo'], out[f'pos_{tag}_hi'] = (float(x) for x in np.percentile(pos, [2.5, 97.5]))
        out[f'roll_{tag}'] = float(pooled.mean())
        out[f'roll_{tag}_lo'], out[f'roll_{tag}_hi'] = (float(x) for x in np.percentile(pooled, [2.5, 97.5]))
    # corectia deplasarii: E[cov_hat] ~ g1 - (g0 + 2 g1) / n  =>  g1 ~ (cov_mediu + var_medie / n) / (1 - 2 / n)
    g1 = (E['cov'].mean() + v / n) / (1 - 2 / n)
    out['roll_corr'] = float(1e4 * 2 * np.sqrt(max(-g1, 0)))
    out['roll5'] = float(1e4 * 2 * np.sqrt(max(-E['cov'].mean(), 0)))
    return out


def bet_etf_pair(start=PD_START):
    """Logaritmul preturilor de inchidere: ETF-ul Patria-TVBETETF (zilele cu tranzactii) si indicele BET-TR, zile comune."""
    e = read_market('TVBETETF.RO')
    e = e[e['volume'] > 0]
    idx = read_market('BETTR.INDX')
    j = pd.concat([e['close'].rename('etf'), idx['close'].rename('idx')], axis=1, join='inner').dropna().loc[start:END_PD]
    return np.log(j)

## Analysis

In [ ]:
def boot_days(values_by_day, stat, B=B, seed=SEED):
    """Bootstrap pe zile (blocuri = zile intregi): interval percentil de 95% pentru stat(esantion de zile)."""
    rng = np.random.default_rng(seed)
    days = np.array(list(values_by_day.keys()))
    out = []
    for _ in range(B):
        pick = rng.choice(days, len(days), replace=True)
        out.append(stat([values_by_day[d] for d in pick]))
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))


def a1_roll(var=0.0520, cov=-0.0081, price=20.0, rho_q=0.3):
    """Modelul Roll: momentele, estimatorul si deplasarea cand semnele tranzactiilor sunt autocorelate.
    Cu Corr(q_t, q_{t-1}) = rho_q (lant Markov, m independent de q): Cov(dp_t, dp_{t-1}) = -c^2 (1 - rho_q)^2."""
    c = np.sqrt(-cov)
    cov_rho = -c ** 2 * (1 - rho_q) ** 2
    return dict(c=c, s=2 * c, s_pct=100 * 2 * c / price, sig2=var - 2 * c ** 2, share=2 * c ** 2 / var,
                rho=cov / var, var=var, cov=cov, price=price, rho_q=rho_q, cov_rho=cov_rho,
                c_roll_rho=np.sqrt(-cov_rho), bias_pct=100 * rho_q)


def a2_roll_se(var=0.0520, cov=-0.0081, Ts=(78, 250)):
    """Eroarea standard a lui s_hat = 2 sqrt(-cov_hat) prin metoda delta; Var(cov_hat) din formula lui Bartlett
    pentru un MA(1) (aproximare gaussiana): T Var(cov_hat) -> g0^2 + 3 g1^2."""
    from scipy.stats import norm
    c = np.sqrt(-cov)
    out = dict(var=var, cov=cov, s=2 * c, avar=var ** 2 + 3 * cov ** 2)
    for T in Ts:
        se_g = np.sqrt((var ** 2 + 3 * cov ** 2) / T)
        se_s = se_g / c                               # ds/dg = -1 / sqrt(-g) = -1 / c
        out[T] = dict(se_g=se_g, se_s=se_s, cv=se_s / (2 * c), p_pos=norm.cdf(cov / se_g))
    return out


def gm_spread_formula(theta, mu, dv=20.0):
    """Spread-ul Glosten-Milgrom pentru theta general: a - b = 4 mu theta (1 - theta) dV / (1 - mu^2 (2 theta - 1)^2)."""
    return 4 * mu * theta * (1 - theta) * dv / (1 - mu ** 2 * (2 * theta - 1) ** 2)


def a3_gm(mu=0.3, theta=0.5, vl=90.0, vh=110.0):
    ask, bid, tb, ts = gm_quotes(theta, mu, vl, vh)
    ask2, bid2, tb2, ts2 = gm_quotes(tb, mu, vl, vh)          # dupa o cumparare
    a7, b7, _, _ = gm_quotes(0.7, mu, vl, vh)
    return dict(ask=ask, bid=bid, spread=ask - bid, pb_h=mu + (1 - mu) / 2, pb_l=(1 - mu) / 2, th_b=tb, th_s=ts,
                ask2=ask2, bid2=bid2, spread2=ask2 - bid2, th_b2=tb2, spread_07=a7 - b7,
                formula_07=gm_spread_formula(0.7, mu, vh - vl), formula_05=gm_spread_formula(0.5, mu, vh - vl))


def a4_gm_learning(mus=(0.1, 0.3, 0.5), level=108.0, vl=90.0, vh=110.0):
    """Dupa k cumparari consecutive, cota P(V_H) se inmulteste cu (1 + mu) / (1 - mu) la fiecare cumparare;
    bid-ul dupa k cumparari este V_L + dV theta_{k-1}, deci bid > level cere (k - 1) ln((1 + mu)/(1 - mu)) > ln(o*)."""
    q = (level - vl) / (vh - vl)
    out = {}
    for mu in mus:
        k = 1 + int(np.floor(np.log(q / (1 - q)) / np.log((1 + mu) / (1 - mu)))) + 1
        theta, kk = 0.5, 0
        while True:                                         # verificare directa cu cotatiile
            kk += 1
            ask, bid, tb, ts = gm_quotes(theta, mu, vl, vh)
            if bid > level:
                break
            theta = tb
        out[mu] = dict(k=kk - 1, k_formula=k, speed=np.log((1 + mu) / (1 - mu)))   # kk - 1 cumparari observate
    return out


def a5_kyle(sigma_v=2.0, sigma_u=10_000.0, y=15_000.0, sigma_u2=20_000.0):
    k = kyle(sigma_v, sigma_u)
    k2 = kyle(sigma_v, sigma_u2)
    k.update(sigma_v=sigma_v, sigma_u=sigma_u, y=y, dp=k['lam'] * y, lam100=100 * k['lam'],
             lam2=k2['lam'], profit2=k2['profit'], beta2=k2['beta'])
    return k


def a6_kyle_ols(sigma_v=2.0, sigma_u=10_000.0, p0=50.0, n=200_000, seed=SEED):
    """Economia Kyle simulata: panta OLS a lui (p - p0) pe y este lambda; un raport de tip Amihud |r| / (p0 |y|)
    estimeaza lambda / p0^2, iar cu volumul total |x| + |u| in loc de |y| il subestimeaza."""
    rng = np.random.default_rng(seed)
    k = kyle(sigma_v, sigma_u)
    v = p0 + sigma_v * rng.standard_normal(n)
    x = k['beta'] * (v - p0)
    u = sigma_u * rng.standard_normal(n)
    y = x + u
    p = p0 + k['lam'] * y
    slope = np.polyfit(y, p - p0, 1)[0]
    slope_v = np.polyfit(y, v - p0, 1)[0]
    r = np.abs(p - p0) / p0
    am_net = np.mean(r / (p0 * np.abs(y)))
    am_vol = np.mean(r / (p0 * (np.abs(x) + np.abs(u))))
    return dict(lam=k['lam'], slope=slope, slope_v=slope_v, target=k['lam'] / p0 ** 2, am_net=am_net, am_vol=am_vol,
                am_ratio=am_vol / am_net, p0=p0, var_y=np.var(y), var_y_theory=2 * sigma_u ** 2)


def a7_ac(lam=2e-6):
    """Almgren-Chriss: solutia in timp continuu (Euler-Lagrange: x'' = kappa^2 x) si programul discret exact."""
    tau = AC['T'] / AC['N']
    eta_t = AC['eta'] - AC['gamma'] * tau / 2
    kt2 = lam * AC['sigma'] ** 2 / eta_t
    kappa = np.arccosh(kt2 * tau ** 2 / 2 + 1) / tau
    kc = np.sqrt(lam * AC['sigma'] ** 2 / AC['eta'])          # timp continuu: eta_tilde -> eta cand tau -> 0
    t, x, E, V = ac_trajectory(lam=lam, **AC)
    t0, x0, E0, V0 = ac_trajectory(lam=0, **AC)
    xc = AC['X'] * np.sinh(kc * (AC['T'] - t)) / np.sinh(kc * AC['T'])
    return dict(lam=lam, eta_t=eta_t, kt2=kt2, kappa=kappa, kc=kc, kc2=kc ** 2, x=list(x), xc=list(xc),
                n=list(-np.diff(x)), E=E, sd=np.sqrt(V), E0=E0, sd0=np.sqrt(V0), sinh5=np.sinh(5 * kappa))


def a8_ac():
    return a7_ac(lam=2e-5)


def pin_posterior(B, S, alpha, delta, mu, eb, es):
    """Probabilitatile a posteriori ale celor trei stari (fara stire, stire proasta, stire buna) dupa (B, S)."""
    from scipy.special import softmax
    _, terms = pin_loglik(B, S, alpha, delta, mu, eb, es)
    return softmax(terms)


def a9_pin(B=1450, S=1000, **par):
    """Verosimilitatea EKOP pentru o zi: evaluarea directa esueaza numeric, forma factorizata nu."""
    import warnings as _w
    par = par or PIN_PAR
    a, d, mu, eb, es = (par[k] for k in ('alpha', 'delta', 'mu', 'eb', 'es'))
    pin = a * mu / (a * mu + eb + es)
    with _w.catch_warnings():
        _w.simplefilter('ignore')
        naive = np.exp(-eb) * np.float64(eb) ** B                 # e^-1000 = 0 si 1000^1450 = inf: 0 * inf = nan
    ll, _ = pin_loglik(B, S, a, d, mu, eb, es)
    post = pin_posterior(B, S, a, d, mu, eb, es)
    return dict(pin=pin, naive=str(naive), exp_small=float(np.exp(-eb)), loglik=ll, p_none=post[0], p_bad=post[1],
                p_good=post[2], B=B, S=S, **par)


def a10_pin():
    """Aceiasi parametri: o zi cu stire proasta probabila (B = 1000, S = 1420), o zi linistita (B = S = 1000)
    si PIN cand mu se dubleaza."""
    x = a9_pin(B=1000, S=1420)
    q = a9_pin(B=1000, S=1000)
    par = dict(PIN_PAR, mu=800.0)
    pin2 = par['alpha'] * par['mu'] / (par['alpha'] * par['mu'] + par['eb'] + par['es'])
    return dict(bad=dict(p_none=x['p_none'], p_bad=x['p_bad'], p_good=x['p_good'], loglik=x['loglik']),
                quiet=dict(p_none=q['p_none'], p_bad=q['p_bad'], p_good=q['p_good']), pin=x['pin'], pin2=pin2)


def b1_ushape():
    s = intraday_spy()
    x = s.dropna(subset=['r'])
    by_day = {}
    for d, g in x.groupby('date'):
        g = g.set_index('tod')
        by_day[d] = (abs(g['r'].get('09:30', np.nan)), g.loc['11:30':'14:00', 'r'].abs().mean(),
                     g['volume'].iloc[-6:].sum() / g['volume'].sum())
    arr = np.array(list(by_day.values()))
    ratio = np.nanmean(arr[:, 0]) / np.nanmean(arr[:, 1])
    lo, hi = boot_days(by_day, lambda v: np.nanmean([a[0] for a in v]) / np.nanmean([a[1] for a in v]))
    vs = np.nanmean(arr[:, 2])
    vlo, vhi = boot_days(by_day, lambda v: np.nanmean([a[2] for a in v]))
    tod = spy_tod(s)
    # grafic: media |r| pe intervale, cu benzi bootstrap pe zile
    piv = x.pivot_table(index='date', columns='tod', values='r', aggfunc=lambda v: abs(v).mean())
    rng = np.random.default_rng(SEED)
    bs = np.array([piv.iloc[rng.integers(0, len(piv), len(piv))].mean().values for _ in range(300)])
    lo_b, hi_b = 1e4 * np.nanpercentile(bs, 2.5, axis=0), 1e4 * np.nanpercentile(bs, 97.5, axis=0)
    k = np.arange(len(piv.columns))
    fig, ax = plt.subplots(figsize=(8.4, 3.2))
    ax.fill_between(k, lo_b, hi_b, color='#DADADA', alpha=0.8, lw=0, label='95% bootstrap band (resampling days)')
    ax.plot(k, 1e4 * piv.mean().values, color=IDAred, lw=1.2, marker='o', ms=2.5, label='Mean |5-minute return| (bp)')
    ax.set_xticks(k[::12])
    ax.set_xticklabels(piv.columns[::12], fontsize=7.5)
    ax.set_xlabel('Start of the 5-minute bar (New York time)')
    ax.set_ylabel('Mean |return| (bp)')
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig('ch10_sem_ushape')
    return dict(ratio=ratio, lo=lo, hi=hi, open=1e4 * np.nanmean(arr[:, 0]), mid=1e4 * np.nanmean(arr[:, 1]),
                vshare=100 * vs, vlo=100 * vlo, vhi=100 * vhi, n=len(by_day), absr_close=float(tod['absr'].iloc[-1]))


def b2_spreads():
    s = intraday_spy()
    E = intraday_spreads(s)
    by_day = {d: row for d, row in E.iterrows()}
    roll5 = 1e4 * 2 * np.sqrt(max(-E['cov'].mean(), 0))
    rlo, rhi = boot_days(by_day, lambda v: 1e4 * 2 * np.sqrt(max(-np.mean([r['cov'] for r in v]), 0)))
    cs5 = 1e4 * E['cs'].mean()
    clo, chi = boot_days(by_day, lambda v: 1e4 * np.mean([r['cs'] for r in v]))
    ar5 = 1e4 * np.sqrt(max(E['ar2'].mean(), 0))
    alo, ahi = boot_days(by_day, lambda v: 1e4 * np.sqrt(max(np.mean([r['ar2'] for r in v]), 0)))
    tick = 1e4 * (0.01 / E['price']).mean()
    # date zilnice: bootstrap pe blocuri mobile de 20 de zile
    d = ohlc('SPY', START2)
    cs = cs_spread(d['high'], d['low'])
    ar2 = ar_terms(d['close'], d['high'], d['low'])
    rng = np.random.default_rng(SEED)
    n, L = len(cs), 20
    csb, arb = [], []
    for _ in range(B):
        st = rng.integers(0, n - L, n // L + 1)
        idx = np.concatenate([np.arange(a, a + L) for a in st])[:n]
        csb.append(1e4 * cs[idx].mean())
        arb.append(1e4 * np.sqrt(max(ar2[idx].mean(), 0)))
    sd_daily = 1e4 * np.log(d['close']).diff().std()
    return dict(roll5=roll5, rlo=rlo, rhi=rhi, cs5=cs5, clo=clo, chi=chi, ar5=ar5, alo=alo, ahi=ahi, tick=tick,
                cs_d=1e4 * cs.mean(), cs_dlo=float(np.percentile(csb, 2.5)), cs_dhi=float(np.percentile(csb, 97.5)),
                ar_d=1e4 * np.sqrt(max(ar2.mean(), 0)), ar_dlo=float(np.percentile(arb, 2.5)),
                ar_dhi=float(np.percentile(arb, 97.5)), sd_daily=sd_daily, share_negcov=float((E['cov'] < 0).mean()))


def b3_cross():
    T = spread_table()
    A = amihud_table()
    rows = []
    for k in ASSETS:
        r = returns(k, START2)
        ppy = 365 if ASSETS[k][2] == 'Crypto' else 252
        rows.append(dict(key=k, grp=ASSETS[k][2], cs=T[k]['cs'], ar=T[k]['ar'], vol=100 * r.std() * np.sqrt(ppy),
                         illiq=A[k]['illiq'], dv=A[k]['dv_med']))
    t = pd.DataFrame(rows).set_index('key')
    rho_cs_vol = stats.spearmanr(t['cs'], t['vol'])
    rho_il_vol = stats.spearmanr(t['illiq'], t['vol'])
    rho_cs_il = stats.spearmanr(t['cs'], t['illiq'])
    rho_il_dv = stats.spearmanr(t['illiq'], t['dv'])
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    for g, c in GROUP_COL.items():
        m = t['grp'] == g
        axes[0].scatter(t.loc[m, 'vol'], t.loc[m, 'cs'], color=c, s=22, label=g)
        axes[1].scatter(t.loc[m, 'dv'], t.loc[m, 'illiq'], color=c, s=22)
    axes[0].set_xlabel('Annualised volatility (%)')
    axes[0].set_ylabel('Corwin-Schultz spread (bp)')
    axes[0].set_title(f'Spearman correlation {rho_cs_vol.statistic:.2f}', fontsize=9, loc='left')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Median daily traded value (USD million)')
    axes[1].set_ylabel('Amihud illiquidity (bp per USD 1m)')
    axes[1].set_title(f'Spearman correlation {rho_il_dv.statistic:.2f}', fontsize=9, loc='left')
    h, l = axes[0].get_legend_handles_labels()
    fig_legend_bottom(fig, h, l, ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save_fig('ch10_sem_cross')
    return dict(rho_cs_vol=float(rho_cs_vol.statistic), p_cs_vol=float(rho_cs_vol.pvalue),
                rho_il_vol=float(rho_il_vol.statistic), rho_cs_il=float(rho_cs_il.statistic),
                rho_il_dv=float(rho_il_dv.statistic), p_il_dv=float(rho_il_dv.pvalue),
                n=int(len(t)), n_ar_zero=int((t['ar'] == 0).sum()),
                bvb_cs_med=float(t.loc[t['grp'] == 'BVB', 'cs'].median()), us_cs_med=float(t.loc[t['grp'] == 'US', 'cs'].median()),
                cr_cs_med=float(t.loc[t['grp'] == 'Crypto', 'cs'].median()),
                table=t.round(4).to_dict(orient='index'))


def b4_illiq_vix():
    s = intraday_spy()
    ill = spy_illiq_daily(s)
    vix = read_market('VIX.INDX')['close'].rename('vix')
    j = pd.concat([ill, vix], axis=1, join='inner').dropna()
    y, X = np.log(j['illiq']), sm.add_constant(np.log(j['vix']))
    ols = sm.OLS(y, X).fit()
    hac = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
    # control pentru nivelul valorii tranzactionate (tendinta)
    dvd = s.groupby('date')['dv'].sum().rename('dv')
    j2 = pd.concat([j, dvd], axis=1, join='inner').dropna()
    X2 = sm.add_constant(pd.DataFrame({'lvix': np.log(j2['vix']), 't': np.arange(len(j2)) / 252}))
    hac2 = sm.OLS(np.log(j2['illiq']), X2).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
    rho1 = float(np.corrcoef(ols.resid[1:], ols.resid[:-1])[0, 1])
    # (d) stabilitatea elasticitatii: ruptura la mijlocul esantionului (data fixata inainte de estimare), test Wald HAC
    mid = j.index[len(j) // 2]
    post = (j.index >= mid).astype(float)
    X3 = sm.add_constant(pd.DataFrame({'lvix': np.log(j['vix']), 'post': post, 'lvix_post': post * np.log(j['vix'])},
                                      index=j.index))
    hac3 = sm.OLS(y, X3).fit(cov_type='HAC', cov_kwds={'maxlags': 20})
    w = hac3.wald_test('lvix_post = 0', scalar=True)
    return dict(b=float(hac.params.iloc[1]), se_ols=float(ols.bse.iloc[1]), se_hac=float(hac.bse.iloc[1]),
                brk=mid.strftime('%Y-%m-%d'), b_pre=float(hac3.params['lvix']),
                b_post=float(hac3.params['lvix'] + hac3.params['lvix_post']), d_brk=float(hac3.params['lvix_post']),
                se_brk=float(hac3.bse['lvix_post']), p_brk=float(w.pvalue),
                lo=float(hac.params.iloc[1] - 1.96 * hac.bse.iloc[1]), hi=float(hac.params.iloc[1] + 1.96 * hac.bse.iloc[1]),
                r2=float(ols.rsquared), n=int(len(j)), rho1=rho1, b2=float(hac2.params['lvix']),
                se2=float(hac2.bse['lvix']), trend=float(hac2.params['t']), se_trend=float(hac2.bse['t']))


def b5_amihud_groups():
    per = {}
    for a, b in [('2024-09-19', '2025-09-18'), ('2025-09-19', '2026-09-18')]:
        per[a[:4]] = {k: float((1e4 * returns(k, a, b).abs() / (dollar_volume(k, a, b) / 1e6)).dropna().mean())
                      for k in ASSETS}
    p1, p2 = pd.Series(per['2024']), pd.Series(per['2025'])
    rho = stats.spearmanr(p1, p2)
    A = amihud_table()
    il = pd.Series({k: v['illiq'] for k, v in A.items()})
    grp = pd.Series({k: v[2] for k, v in ASSETS.items()})
    rng = np.random.default_rng(SEED)
    ratios = []
    for _ in range(B):
        bb = il[grp == 'BVB'].sample(frac=1, replace=True, random_state=rng.integers(1e9)).median()
        uu = il[grp == 'US'].sample(frac=1, replace=True, random_state=rng.integers(1e9)).median()
        ratios.append(bb / uu)
    return dict(rho=float(rho.statistic), p=float(rho.pvalue), med_bvb=float(il[grp == 'BVB'].median()),
                med_us=float(il[grp == 'US'].median()), med_cr=float(il[grp == 'Crypto'].median()),
                ratio=float(il[grp == 'BVB'].median() / il[grp == 'US'].median()),
                lo=float(np.percentile(ratios, 2.5)), hi=float(np.percentile(ratios, 97.5)))


def b6_sqrt():
    s = intraday_spy()
    x, g, slope, _ = sqrt_relation(s)
    days = x['date'].unique()
    by = {d: grp for d, grp in x.groupby('date')}
    bins = np.quantile(x['part'], np.linspace(0, 1, 21))
    rng = np.random.default_rng(SEED)
    sl = []
    for _ in range(300):
        pick = rng.choice(days, len(days), replace=True)
        xx = pd.concat([by[d] for d in pick])
        gg = xx.groupby(pd.cut(xx['part'], bins, include_lowest=True)).agg(v=('part', 'mean'), z=('z', 'mean')).dropna()
        sl.append(np.polyfit(np.log(gg['v']), np.log(gg['z']), 1)[0])
    lo, hi = np.percentile(sl, [2.5, 97.5])
    # pe barele individuale (nu pe grupe)
    y = np.log(x['r'].abs() + 1e-6) - np.log(x.groupby('date')['r'].transform('std'))
    raw = sm.OLS(y, sm.add_constant(np.log(x['part']))).fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(x['date'])[0]})
    # (c) variabila instrumentala: volumul relativ al aceleiasi bare (aceeasi ora) din ziua precedenta
    x = x.assign(ly=y, lv=np.log(x['part']))
    x['lv_lag'] = x.groupby('tod')['lv'].shift(1)             # barele sunt ordonate in timp; shift pe ora = ziua anterioara
    x['ly_lag'] = x.groupby('tod')['ly'].shift(1)
    z = x.dropna(subset=['lv_lag', 'ly_lag'])
    gid = pd.factorize(z['date'])[0]
    fs = sm.OLS(z['lv'], sm.add_constant(z['lv_lag'])).fit(cov_type='cluster', cov_kwds={'groups': gid})
    Z = sm.add_constant(z['lv_lag']).values
    Xe = sm.add_constant(z['lv']).values
    Pz = Z @ np.linalg.solve(Z.T @ Z, Z.T @ Xe)               # proiectia regresorilor pe instrumente
    b_iv = np.linalg.solve(Pz.T @ Xe, Pz.T @ z['ly'].values)
    e = z['ly'].values - Xe @ b_iv
    A = np.linalg.inv(Pz.T @ Pz)
    meat = sum(np.outer(Pz[gid == gg].T @ e[gid == gg], Pz[gid == gg].T @ e[gid == gg]) for gg in np.unique(gid))
    se_iv = np.sqrt(np.diag(A @ meat @ A))
    red = np.corrcoef(z['ly'], z['ly_lag'])[0, 1]            # |r| de ieri la aceeasi ora: canalul care incalca excluderea
    return dict(slope=float(slope), lo=float(lo), hi=float(hi), n=int(len(x)), n_days=int(len(days)),
                raw=float(raw.params.iloc[1]), raw_se=float(raw.bse.iloc[1]), fs=float(fs.params.iloc[1]),
                fs_t=float(fs.tvalues.iloc[1]), iv=float(b_iv[1]), iv_se=float(se_iv[1]), n_iv=int(len(z)),
                corr_absr_lag=float(red))


def b7_btc():
    b = intraday_btc().dropna(subset=['r'])
    wk = b.index.dayofweek >= 5
    h = b.index.hour
    by_day = {}
    for d, g in b.groupby('date'):
        hh = g.index.hour
        by_day[d] = (g['r'].abs()[(hh >= 13) & (hh < 17)].mean(), g['r'].abs()[(hh >= 3) & (hh < 11)].mean(),
                     g['r'].abs().mean(), d.dayofweek >= 5)
    wd = {d: v for d, v in by_day.items() if not v[3]}
    ratio = np.nanmean([v[0] for v in wd.values()]) / np.nanmean([v[1] for v in wd.values()])
    lo, hi = boot_days(wd, lambda v: np.nanmean([a[0] for a in v]) / np.nanmean([a[1] for a in v]))
    wkr = np.nanmean([v[2] for v in by_day.values() if not v[3]]) / np.nanmean([v[2] for v in by_day.values() if v[3]])
    lo2, hi2 = boot_days(by_day, lambda v: np.nanmean([a[2] for a in v if not a[3]]) / np.nanmean([a[2] for a in v if a[3]]))
    return dict(ratio=float(ratio), lo=lo, hi=hi, wk=float(wkr), wklo=lo2, wkhi=hi2, n_days=len(by_day),
                us=float(1e4 * b['r'].abs()[(~wk) & (h >= 13) & (h < 17)].mean()),
                asia=float(1e4 * b['r'].abs()[(~wk) & (h >= 3) & (h < 11)].mean()))


def b8_price_discovery(p=1, B=B, L=20, seed=SEED):
    """ETF-ul Patria-TVBETETF si indicele BET-TR: VECM cu vectorul (1, -1), limitele ponderii Hasbrouck, ponderea
    Gonzalo-Granger; intervale bootstrap pe blocuri mobile (blocuri de L zile) ale perechilor (X_t, Y_t) din VECM;
    apoi cele doua jumatati ale esantionului."""
    y = bet_etf_pair().values
    base = price_discovery(y, p)
    dy = np.diff(y, axis=0)
    z = (y[:, 0] - y[:, 1])[:-1]
    X = np.array([[1.0, z[t]] + [v for i in range(1, p + 1) for v in dy[t - i]] for t in range(p, len(dy))])
    Y = dy[p:]
    rng = np.random.default_rng(seed)
    n = len(Y)
    draws = []
    for _ in range(B):
        st = rng.integers(0, n - L, n // L + 1)
        idx = np.concatenate([np.arange(a, a + L) for a in st])[:n]
        Xb, Yb = X[idx], Y[idx]
        Bb = np.linalg.lstsq(Xb, Yb, rcond=None)[0]
        U = Yb - Xb @ Bb
        Om = U.T @ U / (n - X.shape[1])
        al = Bb[1]
        ap = np.array([-al[1], al[0]])
        ISs = []
        for order in ([0, 1], [1, 0]):
            F = np.linalg.cholesky(Om[np.ix_(order, order)])
            v = (ap[order] @ F) ** 2 / (ap @ Om @ ap)
            ISs.append(v[np.argsort(order)][1])
        draws.append((al[0], al[1], ap[1] / ap.sum(), min(ISs), max(ISs)))
    d = np.array(draws)
    ci = lambda k: [float(x) for x in np.percentile(d[:, k], [2.5, 97.5])]
    half = len(y) // 2
    h1, h2 = price_discovery(y[:half], p), price_discovery(y[half:], p)
    return dict(a_etf=float(base['alpha'][0]), a_idx=float(base['alpha'][1]), t_etf=float(base['t'][0]),
                t_idx=float(base['t'][1]), cs_idx=float(base['cs'][1]), is_idx_lo=float(base['is_lo'][1]),
                is_idx_hi=float(base['is_hi'][1]), ci_a_etf=ci(0), ci_a_idx=ci(1), ci_cs_idx=ci(2), ci_is_lo=ci(3),
                ci_is_hi=ci(4), h1_a_etf=float(h1['alpha'][0]), h1_t_etf=float(h1['t'][0]), h2_a_etf=float(h2['alpha'][0]),
                h2_t_etf=float(h2['t'][0]), h1_a_idx=float(h1['alpha'][1]), h2_a_idx=float(h2['alpha'][1]),
                h1_t_idx=float(h1['t'][1]), h2_t_idx=float(h2['t'][1]), corr=base['corr'], n=int(base['n']))


def market_illiq(keys, start='2016-09-19'):
    """Iliciditatea agregata lunara: media transversala a logaritmului Amihud lunar al fiecarui activ."""
    cols = {}
    for k in keys:
        x = pd.concat([returns(k, start).rename('r'), dollar_volume(k, start).rename('dv')], axis=1, join='inner').dropna()
        x = x[x['dv'] > 0]
        m = (1e4 * x['r'].abs() / (x['dv'] / 1e6)).resample('ME').agg(['mean', 'count'])
        cols[k] = np.log(m['mean'].where(m['count'] >= 10))
    return pd.DataFrame(cols).mean(axis=1).rename('lilliq')


def c1_premium():
    out, series = {}, {}
    for name, spec in C_MARKETS.items():
        sym, col = spec['ret']
        p = read_market(sym)[col].loc['2016-08-01':'2026-08-31']
        r = 100 * np.log(p.resample('ME').last()).diff().rename('r')
        li = market_illiq(spec['keys']).dropna()
        # socul de iliciditate: rezidul unui AR(1) pe logaritmul iliciditatii
        ar = sm.OLS(li.iloc[1:].values, sm.add_constant(li.shift(1).iloc[1:].values)).fit()
        shock = pd.Series(ar.resid, index=li.index[1:], name='shock')
        d = pd.concat([r, li.shift(1).rename('lag'), shock], axis=1, join='inner').dropna().loc['2016-11':'2026-08']
        fit = sm.OLS(d['r'], sm.add_constant(d[['lag', 'shock']])).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
        # Amihud si Hurvich (2004): regresia augmentata cu reziduul AR(1) calculat cu phi corectat de deplasare,
        # phi_c = phi + (1 + 3 phi) / T + 3 (1 + 3 phi) / T^2; eroarea standard include incertitudinea lui phi_c
        T_ar = len(li) - 1
        phi = ar.params[1]
        phi_c = min(phi + (1 + 3 * phi) / T_ar + 3 * (1 + 3 * phi) / T_ar ** 2, 0.9999)   # trunchiat la 0.9999
        psi_c = np.mean(li.values[1:] - phi_c * li.values[:-1])
        vc = pd.Series(li.values[1:] - psi_c - phi_c * li.values[:-1], index=li.index[1:], name='vc')
        dc = pd.concat([r, li.shift(1).rename('lag'), vc], axis=1, join='inner').dropna().loc['2016-11':'2026-08']
        fc = sm.OLS(dc['r'], sm.add_constant(dc[['lag', 'vc']])).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
        var_phic = (1 + 3 / T_ar + 9 / T_ar ** 2) ** 2 * ar.bse[1] ** 2
        se_c = float(np.sqrt(fc.params['vc'] ** 2 * var_phic + fc.bse['lag'] ** 2))
        ah = dict(phi_c=float(phi_c), b_lag_c=float(fc.params['lag']), se_lag_c=se_c, t_lag_c=float(fc.params['lag'] / se_c),
                  bias=float(fit.params['lag'] - fc.params['lag']))
        out[name] = dict(n=int(len(d)), b_lag=float(fit.params['lag']), se_lag=float(fit.bse['lag']),
                         t_lag=float(fit.tvalues['lag']), b_shock=float(fit.params['shock']),
                         se_shock=float(fit.bse['shock']), t_shock=float(fit.tvalues['shock']), r2=float(fit.rsquared),
                         phi=float(ar.params[1]), start=d.index[0].strftime('%Y-%m'), end=d.index[-1].strftime('%Y-%m'), **ah)
        series[name] = d
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
    for ax, (name, d) in zip(axes, series.items()):
        c = GROUP_COL[name]
        ax.scatter(d['shock'], d['r'], s=12, color=c, label=f'{name}: monthly observations')
        xs = np.linspace(d['shock'].min(), d['shock'].max(), 20)
        f = sm.OLS(d['r'], sm.add_constant(d['shock'])).fit()
        ax.plot(xs, f.params.iloc[0] + f.params.iloc[1] * xs, color='black', lw=1.0)
        ax.axhline(0, color=Gray, lw=0.5)
        ax.set_xlabel('Illiquidity shock (AR(1) residual of log illiquidity)')
        ax.set_title(name, fontsize=9, loc='left', color=c)
    axes[0].set_ylabel('Market return in the same month (%)')
    h = [a.get_legend_handles_labels()[0][0] for a in axes]
    fig_legend_bottom(fig, h, [a.get_legend_handles_labels()[1][0] for a in axes] , ncol=3, y=0.02)
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    save_fig('ch10_sem_premium')
    # sectiunea transversala BVB: portofoliul celor mai putin lichide vs cele mai lichide (reechilibrare anuala)
    keys = C_MARKETS['BVB']['keys']
    px = pd.concat({k: read_market(ASSETS[k][0])['adjusted_close'] for k in keys}, axis=1).loc['2017-01-01':'2026-08-31']
    mret = np.log(px.resample('ME').last()).diff()
    spreads = []
    for y in range(2018, 2027):
        il = {}
        for k in keys:
            rr = returns(k, f'{y - 1}-01-01', f'{y - 1}-12-31')
            dv = dollar_volume(k, f'{y - 1}-01-01', f'{y - 1}-12-31')
            if len(rr) > 150:
                il[k] = float((rr.abs() / dv).dropna().mean())
        if len(il) < 6:
            continue
        srt = sorted(il, key=il.get)
        liq, ill = srt[:len(srt) // 2], srt[-(len(srt) // 2):]
        yr = mret.loc[str(y)]
        spreads.append((yr[ill].mean(axis=1) - yr[liq].mean(axis=1)).dropna())
    ls = 100 * pd.concat(spreads)
    tt = sm.OLS(ls, np.ones(len(ls))).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
    out['bvb_ls'] = dict(mean=float(ls.mean()), se=float(tt.bse.iloc[0]), t=float(tt.tvalues.iloc[0]), n=int(len(ls)),
                         ann=float(12 * ls.mean()))
    return out

## Run

In [ ]:
for f in [a1_roll, a2_roll_se, a3_gm, a4_gm_learning, a5_kyle, a6_kyle_ols, a7_ac, a8_ac, a9_pin, a10_pin]:
    print(f.__name__, f())
print('B1', b1_ushape())
print('B2', b2_spreads())
print('B3', {k: v for k, v in b3_cross().items() if k != 'table'})
print('B2 (d)', roll_small_sample(intraday_spreads(intraday_spy())))
print('B4', b4_illiq_vix())
print('B5', b5_amihud_groups())
print('B6', b6_sqrt())
print('B7', b7_btc())
print('B8', b8_price_discovery())
print('C1', c1_premium())

![ch10_sem_ushape](./ch10_sem_ushape.png)

Output: `ch10_sem_ushape.pdf`

![ch10_sem_cross](./ch10_sem_cross.png)

Output: `ch10_sem_cross.pdf`

![ch10_sem_premium](./ch10_sem_premium.png)

Output: `ch10_sem_premium.pdf`